In [106]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

In [107]:
train = pd.read_csv('./data/train_oil.csv')
test = pd.read_csv('./data/oil_test.csv')

In [108]:
train = train[train['Onshore/Offshore'] != 'ONSHORE-OFFSHORE']

y = train['Onshore/Offshore'].copy()
y = y.map(lambda x: 1 if x == 'ONSHORE' else 0).values 

X = train.drop(columns=['Onshore/Offshore', 'Field name'], errors='ignore')
X_test = test.drop(columns=['Field name'], errors='ignore')

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

In [109]:
num_imputer = SimpleImputer(strategy='median')
num_scaler = StandardScaler()

X[num_cols] = num_imputer.fit_transform(X[num_cols])
X[num_cols] = num_scaler.fit_transform(X[num_cols])

X_test[num_cols] = num_imputer.transform(X_test[num_cols])
X_test[num_cols] = num_scaler.transform(X_test[num_cols])

In [110]:
cat_imputer = SimpleImputer(strategy='constant', fill_value='none')
cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])
X_cat = cat_encoder.fit_transform(X[cat_cols])

X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])
X_test_cat = cat_encoder.transform(X_test[cat_cols])

In [111]:
X_num = X[num_cols].values
X_test_num = X_test[num_cols].values
X_processed = np.hstack([X_num, X_cat])
X_test_processed = np.hstack([X_test_num, X_test_cat])

In [112]:
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_processed, y,
    test_size=0.2,
    random_state=42
)

In [113]:
X_train_t = torch.tensor(X_train_np, dtype=torch.float32)
y_train_t = torch.tensor(y_train_np, dtype=torch.long) 

X_val_t = torch.tensor(X_val_np, dtype=torch.float32)
y_val_t = torch.tensor(y_val_np, dtype=torch.long)

train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [114]:
class OilNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )
    
    def forward(self, x):
        return self.network(x)

In [115]:
input_dim = X_train_t.shape[1]
model = OilNN(input_dim)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10

In [116]:
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            val_outputs = model(X_batch)
            val_loss += criterion(val_outputs, y_batch).item()
    
    print(f"Epoch {epoch}: train_loss={epoch_loss/len(train_loader):.4f}, val_loss={val_loss/len(val_loader):.4f}")

Epoch 0: train_loss=0.6219, val_loss=0.6547
Epoch 1: train_loss=0.4994, val_loss=0.6193
Epoch 2: train_loss=0.4107, val_loss=0.5538
Epoch 3: train_loss=0.3222, val_loss=0.4720
Epoch 4: train_loss=0.2402, val_loss=0.3899
Epoch 5: train_loss=0.1606, val_loss=0.3246
Epoch 6: train_loss=0.1079, val_loss=0.2931
Epoch 7: train_loss=0.0725, val_loss=0.2968
Epoch 8: train_loss=0.0589, val_loss=0.3089
Epoch 9: train_loss=0.0439, val_loss=0.3384


In [117]:
X_test_t = torch.tensor(X_test_processed, dtype=torch.float32)
test_dataset = TensorDataset(X_test_t)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model.eval()
predictions = []

with torch.no_grad():
    for (X_batch,) in test_loader:
        outputs = model(X_batch)

        preds = torch.argmax(outputs, dim=1) 
        predictions.extend(preds.cpu().numpy())

submission = pd.DataFrame({
    'index': test.index, 
    'Onshore/Offshore': predictions
})
submission.to_csv('./submission.csv', index=False)